# YOLO26 Monocular Depth Estimation: A Hands-On Capability Tour

Ultralytics YOLO26 now ships **monocular depth estimation** as a first-class task. From a single RGB image it predicts a dense, per-pixel distance map in **real meters**, using the same package and workflow as detection, segmentation, and pose. This notebook is a practical tour of what that unlocks, with runnable code for every demo.

**What you will learn**
- Run YOLO26 depth on an image and read metric depth in meters
- Visualize depth across many different scenes
- Build real applications: per-object distance, near/mid/far free-space zones, depth-based background blur, and single-image 3D point clouds
- Measure person-to-person spacing in 3D
- Run depth on video, frame by frame, with input and output clips you can watch
- Fine-tune YOLO26 depth on your own data and export to ONNX

Everything here runs on any machine with a GPU, including a free Colab T4 (set the runtime to a GPU before you start). All paths are relative to the folder the notebook runs in, so nothing is tied to Colab.

## 1. Setup

We install Ultralytics, confirm the environment, and fetch the sample images and clips. The release is pinned to the one every figure and number in this post was produced with, so the results reproduce; newer releases can change inference details such as the input letterboxing, which shifts the metric values slightly. YOLO26 depth needs a recent Ultralytics release, and a GPU runtime keeps inference and the later fine-tune fast.

In [ ]:
!pip install -q "ultralytics==8.4.174"

In [ ]:
import ultralytics, torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())

All the sample media for this notebook lives in one `assets` folder in the companion GitHub repository. The cell below downloads it once into a local `assets/` folder next to the notebook, and every later cell reads from there. If you already cloned the repository, the files are picked up as they are and nothing is downloaded.

In [ ]:
import os, urllib.request

ASSETS_URL = "https://raw.githubusercontent.com/oliverhale07-dot/learnopencv/master/Yolo26-depth-estimation/assets/"
ASSETS = ["bus.jpg", "zidane.jpg",
          "gallery_scene1.jpg", "gallery_scene2.jpg", "gallery_scene3.jpg", "gallery_scene4.jpg",
          "bokeh_car.jpg", "bokeh_pelican.jpg",
          "people.mp4", "aisle.mp4", "car.mp4"]

os.makedirs("assets", exist_ok=True)
for name in ASSETS:
    path = os.path.join("assets", name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(ASSETS_URL + name, path)
print(sorted(os.listdir("assets")))

## 2. Depth from a single image

Loading a depth model is identical to any other YOLO task. The prediction returns a `DepthMap` whose `.data` is a dense tensor shaped like the image, where every value is an absolute distance in meters. This is metric depth, not a relative 0 to 1 map, so you can read real distances straight off it.

In [ ]:
from ultralytics import YOLO
import numpy as np, cv2, matplotlib.pyplot as plt

m = YOLO("yolo26n-depth.pt")
r = m("assets/bus.jpg")

print(type(r[0]))
print("depth tensor shape:", r[0].depth.data.shape)
dvals = np.squeeze(r[0].depth.data.cpu().numpy())
print("depth range (m): %.2f to %.2f" % (dvals.min(), dvals.max()))

A quick way to see what the model produced is to place the input next to a colorized depth map. Near surfaces get small values and far surfaces get large ones, and the colorbar is in meters.

In [ ]:
depth = np.squeeze(r[0].depth.data.cpu().numpy())
img = r[0].orig_img[..., ::-1]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title("Input RGB")
im = ax[1].imshow(depth, cmap="inferno"); ax[1].axis("off"); ax[1].set_title("YOLO26-depth (meters)")
fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 3. Depth across many scenes

One image is not enough to trust a model. Here we take the mix of indoor and outdoor photos from the assets folder and run depth on all of them at once, so you can see how the predictions behave on streets, interiors, and close-ups. Passing a list of paths runs them as a batch.

In [ ]:
import glob
paths = ["assets/bus.jpg"] + sorted(glob.glob("assets/gallery_scene*.jpg")) + ["assets/zidane.jpg"]
res = m(paths, verbose=False)
fig, axes = plt.subplots(len(res), 2, figsize=(9, 3.4 * len(res)), constrained_layout=True)
for i, rr in enumerate(res):
    d = np.squeeze(rr.depth.data.cpu().numpy())
    axes[i, 0].imshow(rr.orig_img[..., ::-1]); axes[i, 0].axis("off")
    axes[i, 1].imshow(d, cmap="inferno"); axes[i, 1].axis("off")
    axes[i, 1].set_title("%.1f - %.1f m" % (d.min(), d.max()))
axes[0, 0].set_title("Input"); axes[0, 1].set_title("Depth")
plt.show()

## 4. Per-object distance

Because the depth is metric, we can attach a real distance to anything a detector finds. We run a standard YOLO26 detector, then read the median depth inside each box and print it on the image. One camera, boxes plus meters, no stereo rig or LiDAR.

In [ ]:
det = YOLO("yolo26n.pt")
ip = "assets/bus.jpg"
dd = np.squeeze(m(ip, verbose=False)[0].depth.data.cpu().numpy())
dr = det(ip, verbose=False)[0]
vis = dr.orig_img.copy()
H, W = dd.shape
for b in dr.boxes:
    x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
    reg = dd[max(0, y1):min(H, y2), max(0, x1):min(W, x2)]
    dist = float(np.median(reg)) if reg.size else 0.0
    name = det.names[int(b.cls[0])]
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 3)
    cv2.putText(vis, "%s %.1fm" % (name, dist), (x1, max(22, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
plt.figure(figsize=(9, 11)); plt.imshow(vis[..., ::-1]); plt.axis("off")
plt.title("Per-object distance in meters (YOLO26 detect + depth)"); plt.show()

## 5. Near, mid and far free-space zones

A simple but useful view for robotics and safety is to paint the scene by distance band. Everything closer than four meters is flagged green, four to eight meters yellow, and beyond eight meters red. This turns a dense depth map into an at-a-glance map of what is close enough to matter.

In [ ]:
d0 = np.squeeze(m("assets/bus.jpg", verbose=False)[0].depth.data.cpu().numpy())
base = cv2.cvtColor(cv2.imread("assets/bus.jpg"), cv2.COLOR_BGR2RGB)
base = cv2.resize(base, (d0.shape[1], d0.shape[0]))
overlay = base.copy()
overlay[d0 < 4] = (0, 255, 0)
overlay[(d0 >= 4) & (d0 < 8)] = (255, 255, 0)
overlay[d0 >= 8] = (255, 0, 0)
blend = cv2.addWeighted(base, 0.55, overlay, 0.45, 0)
plt.figure(figsize=(9, 11)); plt.imshow(blend); plt.axis("off")
plt.title("Free-space zones: green <4m, yellow 4-8m, red >8m"); plt.show()

## 6. Depth-based background blur

Depth also gives us a cheap portrait mode. We blur the whole frame, then blend the sharp original back in for the near pixels and let the blur take over as distance grows. The subject stays crisp while the background softens.

In [ ]:
bokeh_imgs = {"car": "assets/bokeh_car.jpg", "pelican": "assets/bokeh_pelican.jpg"}
fig, ax = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
for row, (nm, path) in enumerate(bokeh_imgs.items()):
    d = np.squeeze(m(path, verbose=False)[0].depth.data.cpu().numpy())
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (d.shape[1], d.shape[0]))
    blurred = cv2.GaussianBlur(rgb, (0, 0), 10)
    dn = (d - d.min()) / (d.max() - d.min() + 1e-6)
    alpha = np.clip((dn - 0.2) / 0.4, 0, 1)[..., None]
    out = (rgb * (1 - alpha) + blurred * alpha).astype(np.uint8)
    ax[row, 0].imshow(rgb); ax[row, 0].axis("off"); ax[row, 0].set_title("Original")
    ax[row, 1].imshow(out); ax[row, 1].axis("off"); ax[row, 1].set_title("Depth-based background blur")

## 7. Single-image 3D point cloud

With a rough focal length we can back-project every pixel into 3D using the pinhole camera model, which lifts a flat photo into a colored point cloud. It is an approximation, not survey-grade geometry, but it shows how metric depth opens the door to 3D reconstruction from one image.

In [ ]:
dp = np.squeeze(m("assets/bus.jpg", verbose=False)[0].depth.data.cpu().numpy())
col = cv2.cvtColor(cv2.imread("assets/bus.jpg"), cv2.COLOR_BGR2RGB)
col = cv2.resize(col, (dp.shape[1], dp.shape[0]))
Hh, Ww = dp.shape
f = 0.8 * Ww
xs, ys = np.meshgrid(np.arange(Ww), np.arange(Hh))
Z = dp
X = (xs - Ww / 2) * Z / f
Y = (ys - Hh / 2) * Z / f
s = 4
Xs = X[::s, ::s].ravel(); Ys = Y[::s, ::s].ravel(); Zs = Z[::s, ::s].ravel()
C = col[::s, ::s].reshape(-1, 3) / 255.0
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(Xs, Zs, -Ys, c=C, s=1)
ax.view_init(elev=-72, azim=-90)
ax.set_xlabel("X"); ax.set_ylabel("depth Z (m)"); ax.set_zlabel("Y")
ax.set_title("Single-image 3D point cloud from YOLO26 depth")
plt.show()

## 8. Person-to-person spacing in 3D

Combining detection and depth lets us estimate how far apart people really are, not just how far apart their boxes look in pixels. We place each person in 3D using the depth at their box, then draw a line between every pair and color it red when they are closer than two meters. This is the basis of crowd spacing and proximity alerts.

In [ ]:
import itertools
ip2 = "assets/bus.jpg"
dd2 = np.squeeze(m(ip2, verbose=False)[0].depth.data.cpu().numpy())
dr2 = det(ip2, verbose=False)[0]
H2, W2 = dd2.shape
f2 = 0.8 * W2
boxes = [[int(v) for v in b.xyxy[0].tolist()] for b in dr2.boxes]
names = [det.names[int(b.cls[0])] for b in dr2.boxes]
pbx = [bx for bx, nm in zip(boxes, names) if nm == "person"]

pts = []
for (x1, y1, x2, y2) in pbx:
    Zc = float(np.median(dd2[max(0, y1):min(H2, y2), max(0, x1):min(W2, x2)]))
    Xc = ((x1 + x2) / 2 - W2 / 2) * Zc / f2
    pts.append((Xc, Zc, x1, y1, x2, y2))

vis2 = dr2.orig_img.copy()
TH = 2.0
for a, bb in itertools.combinations(range(len(pts)), 2):
    da, db = pts[a], pts[bb]
    dist = ((da[0] - db[0]) ** 2 + (da[1] - db[1]) ** 2) ** 0.5
    ca = ((da[2] + da[4]) // 2, (da[3] + da[5]) // 2)
    cb = ((db[2] + db[4]) // 2, (db[3] + db[5]) // 2)
    color = (255, 0, 0) if dist < TH else (0, 180, 0)
    cv2.line(vis2, ca, cb, color, 3)
    mid = ((ca[0] + cb[0]) // 2, (ca[1] + cb[1]) // 2)
    cv2.putText(vis2, "%.1fm" % dist, mid, cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
plt.figure(figsize=(9, 11)); plt.imshow(vis2[..., ::-1]); plt.axis("off")
plt.title("Person-to-person spacing in 3D (red < 2m apart)"); plt.show()

## 9. Depth on video

Depth runs per frame, so any clip becomes a depth clip. The two helpers below do the work: `process_video` reads a clip (optionally from a start frame, keeping every `step`-th frame), runs depth on each frame, and writes a side-by-side input and depth mp4, while `gif_pair` turns the frames into a looping side-by-side GIF, input on the left and depth on the right, that you can watch right here in the notebook. We then run three very different clips: a person walking indoors, a store aisle, and traffic from above.

In [ ]:
import imageio
from IPython.display import Image as IPyImage, display

def process_video(path, out_name, n=80, start=0, step=1):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)      # skip to where the action is
    reads = [cap.read() for _ in range(n * step)]
    cap.release()
    fr = [cv2.resize(f, (640, 360)) for ok, f in reads[::step] if ok]
    h, w = fr[0].shape[:2]
    out = cv2.VideoWriter(out_name, cv2.VideoWriter_fourcc(*"mp4v"), 15, (w * 2, h))
    dm = [np.squeeze(m(f, verbose=False)[0].depth.data.cpu().numpy()) for f in fr]
    for i in range(len(fr)):
        d = dm[i]
        dn = (d - d.min()) / (d.max() - d.min() + 1e-6)
        cmap = (plt.cm.inferno(dn)[:, :, :3] * 255).astype(np.uint8)[:, :, ::-1]
        out.write(np.hstack([fr[i], cmap]))
    out.release()
    return fr, dm

def gif_pair(fr, dm, name, step=2, fps=10):
    sel = list(range(0, len(fr), step))
    ig = [cv2.cvtColor(fr[i], cv2.COLOR_BGR2RGB) for i in sel]
    og = []
    for i in sel:
        d = dm[i]
        dn = (d - d.min()) / (d.max() - d.min() + 1e-6)
        cm = (plt.cm.inferno(dn)[:, :, :3] * 255).astype(np.uint8)
        og.append(np.hstack([cv2.cvtColor(fr[i], cv2.COLOR_BGR2RGB), cm]))
    imageio.mimsave(name + "_input.gif", ig, fps=fps, loop=0)
    imageio.mimsave(name + "_depth.gif", og, fps=fps, loop=0)
    return name + "_input.gif", name + "_depth.gif"

print("clips:", sorted(glob.glob("assets/*.mp4")))

### 9.1 Person walking indoors

Input on the left, depth on the right. Watch the person stay dark (near) against the brighter floor and walls (far) as they move.

In [ ]:
fr_pe, dm_pe = process_video("assets/people.mp4", "depth_people.mp4", 80)
gif_pair(fr_pe, dm_pe, "people")
display(IPyImage("people_depth.gif"))

### 9.2 Store aisle

A fixed camera looks down a retail aisle while two shoppers walk in and browse the shelves. The first ten seconds of the clip are an empty aisle, so we start at frame 600 and keep every third frame of the 60 fps footage. The shoppers read as near, dark shapes while the aisle recedes toward the brighter far end.

In [ ]:
fr_ai, dm_ai = process_video("assets/aisle.mp4", "depth_aisle.mp4", 80, start=600, step=3)
gif_pair(fr_ai, dm_ai, "aisle")
display(IPyImage("aisle_depth.gif"))

### 9.3 Traffic from above

An overhead view is harder, since most of the frame is a flat road, but the vehicle still separates from the ground as it passes.

In [ ]:
fr_car, dm_car = process_video("assets/car.mp4", "depth_car.mp4", 80)
gif_pair(fr_car, dm_car, "car")
display(IPyImage("car_depth.gif"))

## 10. Fine-tune on your own data

If the pretrained model is not accurate enough for your domain, you can fine-tune it. Here we use the tiny built-in `depth8` sample set, which downloads automatically, so the run finishes in seconds and shows the full workflow. For real data you point `data` at your own dataset YAML. A low learning rate with AdamW is the recommended recipe, since the defaults are tuned for training from scratch. The validation metrics to watch are delta1 (higher is better) and abs_rel, rmse and silog (lower is better).

In [ ]:
ft_model = YOLO("yolo26n-depth.pt")
ft = ft_model.train(data="depth8.yaml", epochs=15, imgsz=640,
                    optimizer="AdamW", lr0=1e-4, warmup_bias_lr=1e-4, plots=True)

Training writes a results figure with the loss curves and every validation metric across epochs.

In [ ]:
results_png = sorted(glob.glob("runs/depth/**/results.png", recursive=True))
display(IPyImage(results_png[-1], width=950))

## 11. Export for deployment

Depth models export through the same path as the rest of YOLO26. ONNX is a good portable target, and from there you can go on to TensorRT, CoreML and others for faster edge inference.

In [ ]:
onnx_path = ft_model.export(format="onnx")
print("exported to:", onnx_path)

## Conclusion

YOLO26 folds monocular depth into the same package you already use for detection, segmentation and pose, and because it predicts metric depth you can read real distances from a single camera. That single capability powers a lot: per-object distance, free-space zones, proximity and crowd spacing, depth-based blur, single-image 3D, and per-frame video depth, all shown above. From here, swap in your own images and clips, fine-tune on your domain, and export to the runtime you deploy on.

If this was useful, a good next step is to try the larger `yolo26s/m/l/x-depth` variants for higher accuracy, and to fine-tune on a dataset that matches your camera and scene.